# Ingestão dos dados e criação da camada bronze:

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json

import requests

In [31]:
BRONZE_DIR = Path.cwd().parent / "data" / "bronze"
HEADERS = {"User-Agent": "DataIngestionPipeline/1.0"}

# Intervalo de analise dos dados
START_YEAR = 2015
END_YEAR = 2025

In [ ]:
BRONZE_DIR = Path.cwd().parent / "data" / "bronze"
HEADERS = {"User-Agent": "CESUPA-ETL-Lab/1.0"}

START_YEAR = 2015
END_YEAR = 2025

SUBSISTEMAS = {
    "norte": (-1.46, -48.49),                   # Belém
    "nordeste": (-8.05, -34.88),                # Recife
    "sudeste_centro_oeste": (-15.78, -47.92),   # Brasília
    "sul": (-30.03, -51.23),                    # Porto Alegre
}

In [ ]:
def baixar(url, params=None, tentativas=4, espera_base=1.6, timeout=60):
    """GET com timeout e retry exponencial. Devolve a resposta."""
    import time

    for tentativa in range(1, tentativas + 1):
        try:
            resposta = requests.get(url, params=params, headers=HEADERS, timeout=timeout)
        except requests.RequestException as erro:
            if tentativa == tentativas:
                raise
            print(f"  rede falhou ({type(erro).__name__}); tentativa {tentativa}/{tentativas}")
            time.sleep(espera_base ** tentativa)
            continue

        if resposta.status_code == 200:
            return resposta

        if resposta.status_code in (429, 500, 502, 503, 504):
            espera = espera_base ** tentativa
            print(f"  status {resposta.status_code}; aguardando {espera:.1f}s "
                  f"(tentativa {tentativa}/{tentativas})")
            time.sleep(espera)
            continue

        raise RuntimeError(f"falha definitiva {resposta.status_code} em {resposta.url}: "
                           f"{resposta.text[:200]}")

    raise RuntimeError(f"desisti de {url} após {tentativas} tentativas")


def salvar_bronze(conteudo, fonte, nome, endpoint, **extra):
    """Grava os bytes crus e, ao lado, um .meta.json com de onde vieram e quando."""
    destino = BRONZE_DIR / fonte / nome
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_bytes(conteudo)

    destino.with_suffix(destino.suffix + ".meta.json").write_text(
        json.dumps({
            "_fonte": fonte,
            "_endpoint": endpoint,
            "_extraido_em": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            **extra,
        }, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(f"  bronze gravada: {destino} ({destino.stat().st_size / 1024:.1f} KB)")
    return destino

In [ ]:
url_epe = (
    "https://www.epe.gov.br/sites-pt/publicacoes-dados-abertos/"
    "dados-abertos/Documents/Dados_abertos_Consumo_Mensal.xlsx"
)

print("EPE — baixando planilha de consumo mensal...")
resp_epe = baixar(url_epe, timeout=120)
salvar_bronze(resp_epe.content, "epe_consumo", "consumo_mensal_historico.xlsx", url_epe)